# Hari 001b — Estimasi π dengan Monte Carlo

| Hari | Tanggal | Jenis | Topik |
|------|---------|-------|-------|
| 001b | 2026-11-19 | `sim` | Estimasi π dengan Monte Carlo |

> Bagian dari **100QB** — 100 Days of Quant + Kebun Simulasi.

## Konsep

Metode **Monte Carlo** memperkirakan nilai deterministik dengan mengambil sampel acak berulang kali. Untuk mengestimasi $\pi$:

- Ambil titik acak seragam $(x, y) \in [0, 1]^2$ (kuadran pertama persegi satuan dengan luas $1$).
- Seperempat lingkaran berjari-jari $r = 1$ di kuadran pertama memiliki luas $\frac{1}{4} \pi r^2 = \frac{\pi}{4}$.
- Peluang sebuah titik acak jatuh di dalam seperempat lingkaran ($x^2 + y^2 \le 1$) adalah $\frac{\pi}{4}$.

Dengan demikian, dari $N$ titik acak:

$$\hat{\pi}_N = 4 \times \frac{\text{jumlah titik dengan } x^2 + y^2 \le 1}{N}$$

Berdasarkan **Hukum Bilangan Besar**, $\hat{\pi}_N \to \pi$ saat $N \to \infty$, dengan galat standar menyusut sebanding dengan $\frac{1}{\sqrt{N}}$.

## Setup

In [ ]:
import sys
from pathlib import Path

# Cari root repo supaya package `qb100` di src/ bisa di-import dari mana pun
ROOT = Path.cwd().resolve()
while not (ROOT / "src" / "qb100").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from qb100 import plotting, simulation as sim

plotting.set_style()
rng = sim.get_rng(42)

N_SAMPLES = 20_000
SAMPLE_SIZES = [100, 500, 1_000, 5_000, 10_000, 20_000, 50_000, 100_000]

## Implementasi

In [ ]:
# 1. Bangkitkan N titik acak seragam di [0, 1]^2
pts = rng.random((N_SAMPLES, 2))
x, y = pts[:, 0], pts[:, 1]
print(f"Jumlah titik dibangkitkan: {len(pts):,}")

In [ ]:
# 2. Klasifikasi titik di dalam seperempat lingkaran (x^2 + y^2 <= 1)
inside = (x * x + y * y) <= 1.0
samples = 4.0 * inside.astype(float)
pi_hat = samples.mean()
print(f"Titik di dalam lingkaran : {inside.sum():,} / {N_SAMPLES:,}")
print(f"Estimasi pi              : {pi_hat:.6f}")
print(f"Nilai pi sebenarnya      : {np.pi:.6f}")
print(f"Galat absolut            : {abs(pi_hat - np.pi):.6f}")

In [ ]:
# 3. Ringkasan statistik Monte Carlo (mean, std, standard error)
stats = sim.mc_summary(samples, ci=0.95)
ci_95_low = stats["mean"] - 1.96 * stats["se"]
ci_95_high = stats["mean"] + 1.96 * stats["se"]
print(f"Mean      : {stats["mean"]:.6f}")
print(f"Std       : {stats["std"]:.6f}")
print(f"Std Error : {stats["se"]:.6f}")
print(f"CI 95%    : [{ci_95_low:.6f}, {ci_95_high:.6f}]")

In [ ]:
# 4. Jejak konvergensi estimasi pi dan standard error kumulatif
pi_running = sim.running_mean(samples)
se_running = sim.running_se(samples)
checkpoints = [100, 500, 1_000, 5_000, 10_000, 20_000]
conv_df = pd.DataFrame({
    "N": checkpoints,
    "Estimasi pi": [pi_running[n - 1] for n in checkpoints],
    "Galat Absolut": [abs(pi_running[n - 1] - np.pi) for n in checkpoints],
    "Std Error": [se_running[n - 1] for n in checkpoints],
})
conv_df.round(5)

In [ ]:
# 5. Skala galat terhadap ukuran sampel N (teori ~ 1 / sqrt(N))
rows = []
for n in SAMPLE_SIZES:
    p_sub = sim.get_rng(n).random((n, 2))
    est = 4.0 * ((p_sub[:, 0] ** 2 + p_sub[:, 1] ** 2) <= 1.0).mean()
    rows.append({"N": n, "pi_hat": est, "abs_error": abs(est - np.pi)})
scale_df = pd.DataFrame(rows)
scale_df

In [ ]:
# 6. Distribusi estimasi pi dari 200 eksperimen independen (masing-masing N=5.000)
def _trial_pi(r):
    pts_t = r.random((5_000, 2))
    return 4.0 * ((pts_t[:, 0] ** 2 + pts_t[:, 1] ** 2) <= 1.0).mean()

pi_trials = sim.monte_carlo(_trial_pi, n_sims=200, seed=42)
trial_stats = sim.mc_summary(pi_trials, ci=0.95)
print(f"Rata-rata 200 trial : {trial_stats["mean"]:.5f} (std: {trial_stats["std"]:.5f})")

## Visualisasi

Empat grafik berikut merangkum intuisi geometri, laju konvergensi, skala galat, dan sebaran estimasi $\pi$.

In [ ]:
# Grafik 1: Titik acak di kuadran pertama persegi satuan (2.500 titik pertama)
n_show = 2_500
fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(x[:n_show][inside[:n_show]], y[:n_show][inside[:n_show]], s=8, alpha=0.6, label="Di dalam ($x^2+y^2 \leq 1$)")
ax.scatter(x[:n_show][~inside[:n_show]], y[:n_show][~inside[:n_show]], s=8, alpha=0.6, label="Di luar")

theta = np.linspace(0, np.pi / 2, 200)
ax.plot(np.cos(theta), np.sin(theta), color="black", linewidth=2, label="Busur $r=1$")
ax.set_aspect("equal")
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.set_title(f"Simulasi Monte Carlo $\\pi$ ({n_show:,} titik pertama)")
ax.set_xlabel("$x$")
ax.set_ylabel("$y$")
ax.legend(loc="upper right")
plotting.save_figure(fig, "day001b_pi-scatter")
plt.show()

In [ ]:
# Grafik 2: Konvergensi estimasi pi terhadap jumlah sampel N
n_axis = np.arange(1, N_SAMPLES + 1)
fig, ax = plt.subplots()
ax.plot(n_axis[20:], pi_running[20:], linewidth=1.5, label="Estimasi kumulatif $\\hat{\\pi}_N$")
ax.fill_between(
    n_axis[20:],
    pi_running[20:] - 1.96 * se_running[20:],
    pi_running[20:] + 1.96 * se_running[20:],
    alpha=0.2,
    label="Pita CI 95% ($\\pm 1.96 \\cdot SE$)",
)
ax.axhline(np.pi, color="black", linestyle="--", linewidth=1.2, label="$\\pi$ sebenarnya")
ax.set_xscale("log")
ax.set_ylim(2.7, 3.6)
ax.set_title("Konvergensi Estimasi $\\pi$ dengan Monte Carlo")
ax.set_xlabel("Jumlah Sampel $N$ (skala log)")
ax.set_ylabel("Estimasi $\\hat{\\pi}_N$")
ax.legend(loc="upper right")
plotting.save_figure(fig, "day001b_pi-convergence")
plt.show()

In [ ]:
# Grafik 3: Skala galat absolut vs N pada skala log-log
fig, ax = plt.subplots()
ax.loglog(scale_df["N"], scale_df["abs_error"], "o-", linewidth=1.8, label="Galat absolut $|\\hat{\\pi}_N - \\pi|$")
ref_curve = 1.6 / np.sqrt(scale_df["N"])
ax.loglog(scale_df["N"], ref_curve, "--", color="gray", label="Referensi teoritis $\\propto 1/\\sqrt{N}$")
ax.set_title("Penurunan Galat Monte Carlo terhadap Ukuran Sampel")
ax.set_xlabel("Jumlah Sampel $N$")
ax.set_ylabel("Galat Absolut")
ax.legend()
plotting.save_figure(fig, "day001b_pi-error-scaling")
plt.show()

In [ ]:
# Grafik 4: Sebaran 200 estimasi pi independen (N=5.000 per trial)
fig, ax = plt.subplots()
ax.hist(pi_trials, bins=18, edgecolor="white", alpha=0.85)
ax.axvline(np.pi, color="black", linestyle="--", linewidth=1.8, label=f"$\\pi = {np.pi:.4f}$")
ax.axvline(trial_stats["mean"], color="#d1495b", linestyle="-", linewidth=1.8, label=f"Rata-rata = {trial_stats["mean"]:.4f}")
ax.set_title("Distribusi Estimasi $\\pi$ dari 200 Simulasi Independen")
ax.set_xlabel("Estimasi $\\hat{\\pi}$")
ax.set_ylabel("Frekuensi")
ax.legend()
plotting.save_figure(fig, "day001b_pi-histogram")
plt.show()

## Kesimpulan (3 poin)

1. ...
2. ...
3. ...

## Referensi

- ...